# 06 · Data quality: obfuscation defects and what they limit

Google describes its public GA4 sample as obfuscated: it "contains placeholder values including
`<Other>`, NULL, and `''`" and "has somewhat limited internal consistency". The generator injects the
same kinds of defects at known rates. This notebook measures each one with SQL
(`sql/11_data_quality.sql`), then shows what each would do to the headline numbers if the pipeline
didn't repair it.

In [1]:
import pandas as pd

from ga4_analytics import quality as Q
from ga4_analytics.db import Warehouse

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
wh = Warehouse()
checks = Q.run_checks(wh)

-- 11_data_quality.sql
dry run OK | 15.3 MB from 92 daily file(s) [20201101..20210131], columns: device.operating_system, ecommerce.purchase_revenue_in_usd, ecommerce.transaction_id, event_name, event_params, geo.country, traffic_source.name, user_first_touch_timestamp, user_pseudo_id | local tables: ga4.main.event_items, ga4.main.sessions, ga4.main.users


## Measured defect rates

In [2]:
checks[["check_name", "unit", "numerator", "denominator", "rate", "injected_rate"]]

,check_name,unit,numerator,denominator,rate,injected_rate
0,traffic_source placeholder (<Other> / (data deleted)),users,4055,80000,0.0507,0.0500
1,geo.country = <Other>,users,3135,80000,0.0392,0.0400
2,device.operating_system = <Other>,users,4026,80000,0.0503,0.0500
3,user_first_touch_timestamp NULL,users,762,80000,0.0095,0.0100
4,page_location = <Other>,page_view events,48732,486101,0.1003,0.1000
5,event without ga_session_id,events,8577,856418,0.0100,0.0100
6,transaction_id = (not set),purchase events,341,4299,0.0793,0.0800
7,purchase_revenue_in_usd NULL,purchase events,87,4299,0.0202,0.0200
8,duplicate purchase events,purchase events,81,4299,0.0188,0.0150
9,item_name = (not set),items,5853,195563,0.0299,0.0300


Every injected defect is measured within sampling noise of its injected rate. Duplicates read
1.9% against 1.5% injected: 81 found against about 64 ± 8 expected, about two standard errors,
and both dedup keys behave correctly.

Two checks have no injected rate:

- **`ga_session_number` out of time order (6.4% of users)** was *not* planted. QA found it: the
  generator gives same-day repeat sessions independent random hours, so a user's "session 2" can
  start before "session 1". It's left in place because it's exactly the "limited internal
  consistency" the real export has, and fixing it would silently change every number reported so
  far. It's documented instead.
- **traffic_source vs first-session channel (8.5%)** is about 5 points from placeholders and 3.5
  points from the session-order issue above.

## How each defect is handled and what it limits

In [3]:
checks[["check_name", "rate", "handling", "limits"]]

,check_name,rate,handling,limits
0,traffic_source placeholder (<Other> / (data deleted)),0.0507,Channel taken from session event_params instead; first-touch channel marked Unassigned,Any report built on traffic_source (user acquisition) loses ~5% of users to Unassigned
1,geo.country = <Other>,0.0392,Kept as its own bucket,Country cuts undercount every country; small countries can't be analysed
2,device.operating_system = <Other>,0.0503,Kept as its own bucket,OS-level diagnosis (e.g. iOS vs Android) has a 5% blind spot
3,user_first_touch_timestamp NULL,0.0095,Not used; first session derived from events,Acquisition-date cohorts need the event-derived first session
4,page_location = <Other>,0.1003,Not used in this analysis,"Page-level funnels (landing page, PDP) would lose 10% of views"
5,event without ga_session_id,0.0100,Reattached to nearest tagged event within 30 min (01_events_flat.sql),"Without repair: ~1% of events, including funnel events, drop out of sessions"
6,transaction_id = (not set),0.0793,Dedup falls back to session + basket value,"Order-level joins (refunds, CRM, back office) impossible for 8% of orders"
7,purchase_revenue_in_usd NULL,0.0202,Repaired from the sum of item revenue,"Without repair: revenue understated ~2%, AOV biased"
8,duplicate purchase events,0.0188,Deduplicated in 03_transactions.sql,Without dedup: transactions and revenue overstated ~1.5%
9,item_name = (not set),0.0299,"Kept, set to NULL in event_items",Product/category analysis loses 3% of items; can't rank products precisely


## What the repairs are worth: naive vs modelled numbers

In [4]:
naive = wh.query("""
SELECT
    count(*) FILTER (WHERE event_name = 'purchase')                       AS purchase_events,
    sum(ecommerce.purchase_revenue_in_usd) FILTER (WHERE event_name = 'purchase') AS revenue_naive,
    count(*) FILTER (WHERE event_name = 'session_start')                  AS session_start_events
FROM events
WHERE _TABLE_SUFFIX BETWEEN $start_suffix AND $end_suffix
""").iloc[0]
modelled = wh.query("""
SELECT (SELECT count(*) FROM transactions)              AS transactions,
       (SELECT sum(revenue_usd) FROM transactions)      AS revenue,
       (SELECT count(*) FROM sessions)                  AS sessions
""", dry_run=False).iloc[0]
pd.DataFrame({
    "naive (raw export)": [naive["purchase_events"], naive["revenue_naive"], naive["session_start_events"]],
    "modelled": [modelled["transactions"], modelled["revenue"], modelled["sessions"]],
}, index=["transactions", "revenue (USD)", "sessions"]).assign(
    naive_error=lambda d: d["naive (raw export)"] / d["modelled"] - 1)

dry run OK | 295.3 KB from 92 daily file(s) [20201101..20210131], columns: ecommerce.purchase_revenue_in_usd, event_name


,naive (raw export),modelled,naive_error
transactions,"4,299.0000","4,218.0000",0.0192
revenue (USD),"188,667.9500","189,306.3700",-0.0034
sessions,"136,427.0000","139,187.0000",-0.0198


- **Transactions:** counting purchase events overstates orders by about 2% (duplicates).
- **Revenue:** two errors partly cancel. Duplicates add revenue and NULL revenue removes it, so the
  naive total looks "about right" by accident. It would drift whenever either rate changes.
- **Sessions:** counting `session_start` events undercounts sessions by about 2%.

## The biggest trap: channel from `traffic_source`

In [5]:
channel_views = wh.query("""
SELECT channel AS channel, count(*) AS sessions_by_session_params
FROM sessions GROUP BY 1
""", dry_run=False).merge(wh.query("""
SELECT channel_group(first_user_source, first_user_medium) AS channel,
       count(DISTINCT session_key) AS sessions_by_traffic_source
FROM events_flat GROUP BY 1
""", dry_run=False), on="channel", how="outer").fillna(0).set_index("channel")
channel_views["difference"] = channel_views["sessions_by_traffic_source"] / channel_views["sessions_by_session_params"] - 1
channel_views.sort_values("sessions_by_session_params", ascending=False)

,sessions_by_session_params,sessions_by_traffic_source,difference
channel,,,
Organic Search,"37,077.0000",39423,0.0633
Direct,"29,053.0000",18711,-0.3560
Paid Search,"19,842.0000",21274,0.0722
Organic Social,"12,509.0000",15815,0.2643
Email,"12,242.0000",2597,-0.7879
Display,"11,861.0000",15611,0.3162
Referral,"11,669.0000",13447,0.1524
Affiliates,"4,934.0000",5218,0.0576
Unassigned,0.0000,7091,inf


Using `traffic_source` labels every session with the user's *first* channel. Email and Direct,
which bring people back, lose most of their sessions; Display and Social, which acquire them, gain;
and about 5% of sessions fall into Unassigned because of the placeholders. Any channel report or
attribution model built on `traffic_source` is first-click attribution in disguise.

## How obfuscation limits conclusions on the real export

| Limitation | Consequence | What still holds |
|---|---|---|
| Placeholder values (`<Other>`, NULL, `''`) in dimensions | Segment cuts (country, OS, source) have a blind spot of several percent; small segments can't be analysed | Large-segment comparisons, if the placeholder share is similar across segments |
| `(not set)` transaction ids | Orders can't be joined to refunds, margins or CRM; dedup needs a fallback key | Order counts after dedup |
| Limited internal consistency (e.g. revenue vs item sums, session numbering) | Exact reconciliations fail; identities such as revenue = Σ items can't be assumed | Relative comparisons within the dataset |
| Values are obfuscated, not just masked | Absolute levels (revenue, AOV) are not the store's real numbers; Google says not to compare with the GA demo account | Shapes, ratios and differences between segments; the method itself |
| Cookie-level `user_pseudo_id` | Cross-device journeys split, so attribution under-credits channels that start journeys on another device | Within-device journeys |
| Fixed 3-month window | Journeys are cut at both edges; no year-over-year baseline for holidays | Within-window comparisons |

**Bottom line:** on obfuscated data, trust *relative* findings (mobile vs desktop, which step
breaks, which channel opens vs closes) and present absolute numbers as illustrative. Here the
relative findings are also validated against the planted ground truth, which the real export can't
offer.